# 19 - Lidar, Point Clouds, And ICP

## What / Why / How

**What we are trying to do:** Register two 2D point clouds using a small iterative closest point loop.

**Why this matters:** Lidar mapping, scan matching, and point cloud alignment are core tools for mobile robots and 3D perception.

**How we will do it:** Create a transformed point cloud, repeatedly match nearest neighbors, estimate the best rigid transform, and measure alignment error.

## Goal

Learn the shape of lidar and point cloud registration.

You will implement a small 2D ICP loop:

- Generate a point cloud.
- Transform it.
- Estimate the rigid transform back.

In [ ]:
import math
import random
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

try:
    import matplotlib.pyplot as plt
    HAS_PLOT = True
except ModuleNotFoundError:
    plt = None
    HAS_PLOT = False

np.set_printoptions(precision=3, suppress=True)

def plot_unavailable():
    if not HAS_PLOT:
        print("Install matplotlib to see plots: pip install -r requirements.txt")

In [ ]:
rng = np.random.default_rng(5)
angles = np.linspace(0, 2*np.pi, 80, endpoint=False)
base = np.c_[np.cos(angles), 0.6*np.sin(angles)]
base += rng.normal(0, 0.01, base.shape)

true_theta = np.deg2rad(18)
R_true = np.array([[np.cos(true_theta), -np.sin(true_theta)], [np.sin(true_theta), np.cos(true_theta)]])
t_true = np.array([0.4, -0.2])
moved = base @ R_true.T + t_true
print("created two point clouds")

In [ ]:
def nearest_neighbors(src, dst):
    indexes = []
    for p in src:
        indexes.append(int(np.argmin(np.linalg.norm(dst - p, axis=1))))
    return np.array(indexes)

def best_fit_transform(src, dst):
    src_mean = src.mean(axis=0)
    dst_mean = dst.mean(axis=0)
    X = src - src_mean
    Y = dst - dst_mean
    U, _, Vt = np.linalg.svd(X.T @ Y)
    R = Vt.T @ U.T
    if np.linalg.det(R) < 0:
        Vt[-1] *= -1
        R = Vt.T @ U.T
    t = dst_mean - R @ src_mean
    return R, t

aligned = moved.copy()
total_R = np.eye(2)
total_t = np.zeros(2)
for _ in range(20):
    idx = nearest_neighbors(aligned, base)
    R, t = best_fit_transform(aligned, base[idx])
    aligned = aligned @ R.T + t
    total_R = R @ total_R
    total_t = R @ total_t + t
mean_error = np.mean(np.linalg.norm(aligned - base[nearest_neighbors(aligned, base)], axis=1))
print("mean registration error:", round(float(mean_error), 4))
print("estimated inverse translation:", total_t)

In [ ]:
if HAS_PLOT:
    plt.figure(figsize=(5, 5))
    plt.scatter(base[:, 0], base[:, 1], s=12, label="target")
    plt.scatter(moved[:, 0], moved[:, 1], s=12, alpha=0.3, label="before")
    plt.scatter(aligned[:, 0], aligned[:, 1], s=12, alpha=0.7, label="after ICP")
    plt.axis("equal")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    plot_unavailable()

## Exercises

1. Try a less symmetric point cloud.
2. Add outliers and reject bad matches.
3. Explain why ICP needs a reasonable initial guess.